In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
import numpy as np


# Cargar datos
df = pd.read_csv("data//clientes.csv")

# Filtrar columnas relevantes
cols = ["Total", "Rating", "Customer type", "Gender", "Product line", "Payment"]
df = df[cols].dropna()

# EDA numéricas
print(df[["Total", "Rating"]].describe())

# Distribuciones
sns.histplot(df["Total"], kde=True)
plt.title("Distribución de Total")
plt.show()

sns.histplot(df["Rating"], kde=True)
plt.title("Distribución de Rating")
plt.show()

# EDA categóricas
for col in ["Customer type", "Gender", "Product line", "Payment"]:
    print(f"\n{col}:\n", df[col].value_counts(normalize=True).round(3))
    sns.countplot(data=df, x=col)
    plt.title(f"Distribución de {col}")
    plt.xticks(rotation=45)
    plt.show()




In [ ]:





# Función para embedding individual
def embed_column(df, col, n_components=2):
    vec = CountVectorizer()
    mat = vec.fit_transform(df[col].astype(str))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    return svd.fit_transform(mat)

# Embeddings
embed_customer = embed_column(df, "Customer type")
embed_gender = embed_column(df, "Gender")
embed_product = embed_column(df, "Product line")
embed_payment = embed_column(df, "Payment")

# Escalar numéricas
num_scaled = StandardScaler().fit_transform(df[["Total", "Rating"]])

# Matriz final
X = np.hstack([
    num_scaled,
    embed_customer,
    embed_gender,
    embed_product,
    embed_payment
])

In [ ]:
from sklearn.cluster import KMeans

k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = km.fit_predict(X)
df["cluster_k4"] = labels

In [ ]:
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
df["PC1"] = X_pca[:,0]
df["PC2"] = X_pca[:,1]

# UMAP
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X)
df["UMAP1"] = X_umap[:,0]
df["UMAP2"] = X_umap[:,1]

# Gráficos
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="PC1", y="PC2", hue="cluster_k4", palette="tab10", s=40)
plt.title("Clusters con PCA (k=4)")
plt.show()

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="UMAP1", y="UMAP2", hue="cluster_k4", palette="tab10", s=40)
plt.title("Clusters con UMAP (k=4)")
plt.show()

In [ ]:
# Numéricas
def cluster_numeric_summary(df, cluster_col, num_cols):
    return df.groupby(cluster_col)[num_cols].agg(["count", "mean", "std"]).round(2)

num_summary = cluster_numeric_summary(df, "cluster_k4", ["Total", "Rating"])
print(num_summary)

# Categóricas
def cluster_category_summary(df, cluster_col, cat_cols, top_n=3):
    result = {}
    for cluster in sorted(df[cluster_col].unique()):
        sub = df[df[cluster_col] == cluster]
        summary = {}
        for col in cat_cols:
            vc = sub[col].value_counts(normalize=True).head(top_n)
            summary[col] = "; ".join([f"{idx} ({round(val*100,1)}%)" for idx, val in vc.items()])
        result[cluster] = summary
    return pd.DataFrame(result).T

cat_summary = cluster_category_summary(df, "cluster_k4", ["Customer type", "Gender", "Product line", "Payment"])
print(cat_summary)

In [ ]:
buyer_personas = {
    0: {
        "nombre": "Premium Explorer",
        "perfil": "Alto ticket, productos premium, predominio de miembros y pagos digitales.",
        "campaña": "Acceso VIP a lanzamientos exclusivos. Canal: email personalizado + app."
    },
    1: {
        "nombre": "Everyday Shopper",
        "perfil": "Gasto medio, productos de hogar y alimentos, compras regulares.",
        "campaña": "Ofertas semanales y cupones por volumen. Canal: push + cupones."
    },
    2: {
        "nombre": "Style Seeker",
        "perfil": "Moda y accesorios, gasto medio-bajo, alto rating.",
        "campaña": "Lanzamientos y recomendaciones personalizadas. Canal: redes + email."
    },
    3: {
        "nombre": "Low-Ticket Loyal",
        "perfil": "Compras pequeñas frecuentes, alto rating, preferencia por efectivo.",
        "campaña": "Cupones inmediatos y fidelidad por visitas. Canal: SMS + punto de venta."
    }
}

for cluster, info in buyer_personas.items():
    print(f"\nCluster {cluster} → {info['nombre']}")
    print(f"Perfil: {info['perfil']}")
    print(f"Campaña sugerida: {info['campaña']}")